In [ ]:
# Imports and setup

import subprocess
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import HTML, display, Markdown

# Install gdown if it is not available yet
try:
    import gdown
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "gdown"],
        check=True
    )
    import gdown

try:
    import ipywidgets as widgets
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "ipywidgets"],
        check=True
    )
    import ipywidgets as widgets


FILE_ID = "1lgycXDQapixkoWFxIg48qaYcrNl6uSZM"

# Create the DATA folder if it does not exist yet
DATA_DIR = Path("DATA")
DATA_DIR.mkdir(exist_ok=True)

# Adjust name and extension to your dataset if needed
DATA_FILE = DATA_DIR / "data.csv"

# Progress bar
progress = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Setup:",
    bar_style="info",
    orientation="horizontal"
)

status = widgets.HTML(value="Starting setup...")
display(progress, status)

# Check whether the dataset is already available
progress.value = 20
status.value = "Checking dataset..."

if DATA_FILE.exists():
    progress.value = 100
    progress.bar_style = "success"
    status.value = "✓ Dataset already available."

else:
    progress.value = 40
    status.value = "Downloading dataset..."

    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, output=str(DATA_FILE), quiet=False)
    if not DATA_FILE.exists():
        raise FileNotFoundError(f"Download failed. File not found: {DATA_FILE}")

    progress.value = 80
    status.value = "Finishing download..."

    progress.value = 100
    progress.bar_style = "success"
    status.value = "✓ Download complete."

In [ ]:
# Variable definitions
MIN_AGE = 1
MAX_AGE = 100
REFERENCE_YEAR = 2017
MIN_BIRTH_YEAR = REFERENCE_YEAR - MAX_AGE + 1

In [ ]:
#@title CSV Setup
df = pd.read_csv(DATA_FILE)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

question_columns = [
    'N1', 'N2', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9', 'N10',
    'E1', 'E3', 'E4', 'E5', 'E7', 'E9', 'E10', 'C4', 'A4'
]
missing_markers = {'', 'na', 'n/a', 'null', 'none'}

display(HTML(f"""
<div style="
    display: inline-block;
    padding: 10px 14px;
    margin: 8px 0;
    background: #eaf7ee;
    border-left: 4px solid #198754;
    border-radius: 5px;
    font-size: 14px;
">
    <b>Initial data:</b> {len(df):,} rows · {df.shape[1]} columns
</div>
"""))

### 1. Checking the existing columns and initial data

In [ ]:
display(df.head())
display(df.tail())

display(df.describe())

column_info = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Non-null": df.notna().sum(),
    "Missing": df.isna().sum(),
    "Missing (%)": (df.isna().mean() * 100).round(2)
})

column_info["Status"] = np.where(
    column_info["Missing"] > 0,
    "⚠ Missing values",
    "✓ Complete"
)

display(HTML("<h4 style='margin:16px 0 8px;'>Column overview</h4>"))
display(column_info)

html = '<div style="display:flex; flex-wrap:wrap; gap:18px; align-items:flex-start;">'

for col in ["age", "gender", "hand", "target"]:
    table = (
        df[col]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Count")
        .to_html()
    )

    html += f"""
    <div style="
        min-width: 150px;
        padding: 10px;
        background: #f8fafc;
        border: 1px solid #d9e2ec;
        border-radius: 6px;
    ">
        <div style="margin-bottom:8px; color:#1f4e79; font-weight:700;">
            {col}
        </div>
        {table}
    </div>
    """

html += "</div>"

display(HTML(html))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

fig, (ax1, ax2) = plt.subplots(
    nrows=1, 
    ncols=2, 
    figsize=(20, 5), 
    gridspec_kw={"width_ratios": [1, 2.5]}
)

# 1. Verteilung der Zielklasse
target_counts = df["target"].value_counts()
target_order = target_counts.index
total = len(df)

sns.countplot(data=df, x="target", order=target_order, ax=ax1)

labels = [f"{count}\n({count / total:.1%})" for count in target_counts]
ax1.bar_label(ax1.containers[0], labels=labels)

ax1.set_title("Target Distribution")
ax1.set_xlabel("")
ax1.set_ylabel("Count")

# 2. Durchschnittswerte pro Persönlichkeitstyp
heatmap_data = df.groupby("target")[item_cols].mean()
sns.heatmap(heatmap_data, cmap="viridis", ax=ax2, cbar_kws={"label": "Mean Score"})
ax2.set_title("Average Answer per Personality Type")
ax2.set_xlabel("Question Items")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
age_num = pd.to_numeric(df['age'], errors='coerce')
missing_age = age_num.isna()
zero_age = age_num.eq(0)
three_digit_age = age_num.between(MAX_AGE + 1, 999, inclusive="both")
valid_age = age_num.between(MIN_AGE, MAX_AGE, inclusive="both")
birth_year = age_num.between(MIN_BIRTH_YEAR, REFERENCE_YEAR, inclusive='both')
invalid_age = ~(birth_year | valid_age) & ~missing_age

age_clean = age_num.copy()
age_clean.loc[birth_year] = REFERENCE_YEAR - age_num.loc[birth_year]

def missing_or_zero(series):
    normalized = series.astype('string').str.strip().str.lower()
    return series.isna() | normalized.isin(missing_markers | {'0'})

missing_gender = missing_or_zero(df['gender'])
missing_hand = missing_or_zero(df['hand'])

In [ ]:
# Number of zero values per question

# # errors="coerce" turns non-numeric values into NaN, so they are not counted as zeros
# zero_counts = pd.Series({col: pd.to_numeric(df[col], errors="coerce").eq(0).sum()
#     for col in question_columns
#     if col in df.columns
# }).sort_values(ascending=False)

# zero_counts.loc["Total"] = zero_counts.sum()

# display(Markdown("#### Zero values in the questions"))
# display(zero_counts.to_frame().T)


# Rows with at least one zero in the questions (also used for cleaning in section 3)
question_zero = pd.Series(False, index=df.index)

for col in question_columns:
    if col in df.columns:
        question_zero |= pd.to_numeric(df[col], errors='coerce').eq(0)

zero_rows = df.loc[question_zero].copy()

display(Markdown(f'#### Rows with at least one zero in the questions ({len(zero_rows):,})'))
display(zero_rows)

In [ ]:
df_work = df.copy()
df_work["age"] = age_clean

total_rows = len(df_work)
keep = pd.Series(True, index=df_work.index)

steps = [
    ("age missing", missing_age),
    ("age = 0", zero_age),
    ("age three-digit", three_digit_age),
    ("age otherwise invalid", invalid_age),
    ("gender missing or 0", missing_gender),
    ("hand missing or 0", missing_hand),
    ("0 in question items", question_zero),
]

results = []

for name, criterion in steps:
    removed_now = keep & criterion
    removed_count = removed_now.sum()

    keep &= ~criterion

    remaining = keep.sum()
    removed_pct = removed_count / total_rows * 100
    cumulative_removed = total_rows - remaining
    cumulative_pct = cumulative_removed / total_rows * 100

    results.append({
        "Step": name,
        "Removed in this step": removed_count,
        "Removed (%)": round(removed_pct, 3),
        "Remaining rows": remaining,
        "Cumulative removed": cumulative_removed,
        "Cumulative data loss (%)": round(cumulative_pct, 3),
    })

df_clean = df_work.loc[keep].copy()
df_clean["age"] = df_clean["age"].astype("Int64")

loss_table = pd.DataFrame(results)
display(loss_table)

In [ ]:
rows_deleted = len(df) - len(df_clean)

display(HTML(f"""
<div style="
    display: inline-block;
    padding: 10px 14px;
    margin: 8px 0;
    background: #F88379;
    border-left: 4px solid #811331;
    border-radius: 5px;
    font-size: 14px;
">
    <b>Initial data:</b> {len(df):,} rows · {df.shape[1]} columns<br>
    <b>Cleaned data:</b> {len(df_clean):,} rows · {df_clean.shape[1]} columns<br>
    <b>Deleted data:</b> {rows_deleted:,} rows ({rows_deleted / len(df) * 100:.3f} %)
</div>
"""))

df.to_csv(DATA_DIR / "data_clean.csv", index=False)

# Data Exploration 

### Correlations between the questions

The column names tell us which OCEAN trait a question belongs to (N = Neuroticism, E = Extraversion,
C = Conscientiousness, A = Agreeableness).

The 19 items are not spread evenly across the traits. Ten measure Neuroticism and seven Extraversion, but Conscientiousness and Agreeableness have only one item each (C4, A4). Since Undercontroller is defined precisely by C and A, the information needed to spot that type is largely missing from the data. This explains directly why the model only catches ~20% of Undercontrollers while getting most of the Resilients.

In [ ]:
html = (
    pd.Series([c[0] for c in item_cols])
    .map(traits)
    .value_counts()
    .to_frame("Count")
    .to_html(classes="table table-striped table-hover", border=0)
)

HTML(html)

### Every question against the personality type

A `countplot` per column, split by `target`. For each answer (1–5) we see how the four
personality types are distributed. Where the coloured blocks shift from left to right
between the types, that question separates them — that is the signal the model will learn.

In [ ]:
import textwrap
item_labels = {
    "N1":  "I get stressed out easily.",
    "N2":  "I am relaxed most of the time.",
    "N3":  "I worry about things.",
    "N4":  "I seldom feel blue.",
    "N5":  "I am easily disturbed.",
    "N6":  "I get upset easily.",
    "N7":  "I change my mood a lot.",
    "N8":  "I have frequent mood swings.",
    "N9":  "I get irritated easily.",
    "N10": "I often feel blue.",
    "E1":  "I am the life of the party.",
    "E3":  "I feel comfortable around people.",
    "E4":  "I keep in the background.",
    "E5":  "I start conversations.",
    "E7":  "I talk to a lot of different people at parties.",
    "E9":  "I don't mind being the center of attention.",
    "E10": "I am quiet around strangers.",
    "C4":  "I make a mess of things.",
    "A4":  "I sympathize with others' feelings.",
}

order = df["target"].value_counts().index
n, ncols = len(item_cols), 4
nrows = -(-n // ncols)          # round up

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 4 * nrows), sharey=True)

for i, (ax, col) in enumerate(zip(axes.flat, item_cols)):
    # only the first plot draws a legend — we reuse its colours for the whole grid
    sns.countplot(data=df, x=col, hue="target", hue_order=order, ax=ax, legend=(i == 0))
    title = f"{col} — {item_labels.get(col, '')}"
    ax.set_title("\n".join(textwrap.wrap(title, 34)), fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")

# remove empty axes in the last row
for ax in axes.flat[n:]:
    ax.remove()

# one shared legend for the whole figure
handles, labels = axes.flat[0].get_legend_handles_labels()
axes.flat[0].get_legend().remove()
fig.legend(handles, labels, title="Personality type",
           loc="lower center", bbox_to_anchor=(0.5, -0.01), ncol=4)

fig.suptitle("Answer distribution per question, split by personality type", fontsize=15, y=1.0)
plt.tight_layout(rect=[0, 0.03, 1, 0.985])
plt.show()

In [ ]:
answer_share = (
    df[item_cols]
    .apply(lambda col: col.value_counts(normalize=True))
    .reindex([1, 2, 3, 4, 5])
    .T
)

plt.figure(figsize=(10, 8))
sns.heatmap(answer_share, annot=True, fmt=".0%", cmap="Blues", cbar=False)
plt.title("Share of answers 1-5 per question")
plt.xlabel("Answer (1 = Disagree, 5 = Agree)")
plt.show()

In [ ]:
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Compute correlation matrix
corr = numeric_df.corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="coolwarm", annot=True,fmt=".2f", vmin=-1, vmax=1, center=0, square=True, linewidths=0.5)
plt.title("Correlation Matrix between the questions")
plt.show()

top_pos = (
    pairs.sort_values(ascending=False)
    .head(5)
    .round(2)
    .to_frame("Correlation")
    .to_html()
)

top_neg = (
    pairs.sort_values()
    .head(5)
    .round(2)
    .to_frame("Correlation")
    .to_html()
)

html = f"""
<div style="display: flex; gap: 40px;">
    <div>
        <h4>Strongest positive pairs</h4>
        {top_pos}
    </div>
    <div>
        <h4>Strongest negative pairs</h4>
        {top_neg}
    </div>
</div>
"""

display(HTML(html))

In [ ]:
target_order = df["target"].value_counts().index
plausible_age = df[df["age"].between(13, 100)]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))

# 1. Oben Links: Boxenplot Alter pro Typ
sns.boxenplot(
    data=plausible_age,
    x="target",
    y="age",
    order=target_order,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Age per personality type")
axes[0, 0].set_xlabel("")

# 2. Oben Rechts: Histogramm Alter
sns.histplot(
    data=plausible_age,
    x="age",
    hue="target",
    hue_order=target_order,
    bins=40,
    element="step",
    ax=axes[0, 1],
)
axes[0, 1].set_title("Age — values between 13 and 100")
axes[0, 1].set_ylabel("Number of people")
if axes[0, 1].get_legend():
    axes[0, 1].get_legend().remove()

# 3. Unten Links: Gender Verteilung
sns.countplot(
    data=df,
    x="gender",
    hue="target",
    hue_order=target_order,
    ax=axes[1, 0],
)
axes[1, 0].set_title("gender")

# 4. Unten Rechts: Hand Verteilung
sns.countplot(
    data=df,
    x="hand",
    hue="target",
    hue_order=target_order,
    ax=axes[1, 1],
)
axes[1, 1].set_title("hand")

plt.tight_layout()
plt.show()